In [1]:
%env NX_CUGRAPH_AUTOCONFIG=True
import networkx as nx
from itertools import combinations
from collections import defaultdict, Counter
#import igraph as ig
import pandas as pd
import sqlite3
import random

env: NX_CUGRAPH_AUTOCONFIG=True


/home/anitasun/.local/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [3]:
conn = sqlite3.connect("tiktok_breadth_first.db")
cursor = conn.cursor()

In [4]:
query = """
SELECT hashtag_names
FROM videos
JOIN follow_relations
ON videos.reposter_username = follow_relations.from_username
WHERE follow_relations.to_username = 'kamalahq'
"""

kamalahq_docs = cursor.execute(query).fetchall()

In [16]:
query = """
SELECT hashtag_names
FROM videos
JOIN follow_relations
ON videos.reposter_username = follow_relations.from_username
WHERE follow_relations.to_username = 'teamtrump'
"""

teamtrump_docs = cursor.execute(query).fetchall()

In [5]:
kamalahq_token = Counter()
for doc in kamalahq_docs:
    contents = doc[0].lower().split(',')
    kamalahq_token.update(contents)

In [25]:
teamtrump_token = Counter()
for doc in teamtrump_docs:
    contents = doc[0].lower().split(',')
    teamtrump_token.update(contents)

In [27]:
kamalahq_edge_weights = defaultdict(int)
for doc in kamalahq_docs:
    contents = doc[0].lower().split(',')
    tokens = sorted(set(contents)) # sorted such that edge weight pairs are indexed alphabetically
    for u, v in combinations(tokens, 2):
        kamalahq_edge_weights[(u, v)] += 1

In [28]:
teamtrump_edge_weights = defaultdict(int)
for doc in teamtrump_docs:
    contents = doc[0].lower().split(',')
    tokens = sorted(set(contents)) # sorted such that edge weight pairs are indexed alphabetically
    for u, v in combinations(tokens, 2):
        teamtrump_edge_weights[(u, v)] += 1

In [ ]:
G_k = nx.Graph()
for (u, v), weight in kamalahq_edge_weights.items():
    G_k.add_edge(u, v, weight=weight)

In [29]:
G_t = nx.Graph()
for (u, v), weight in teamtrump_edge_weights.items():
    G_t.add_edge(u, v, weight=weight)

In [10]:
nx.write_weighted_edgelist(G_k, 'kamalahq_hashtag_network.edgelist')

In [ ]:
nx.write_weighted_edgelist(G_t, 'teamtrump_hashtag_network.edgelist')

In [4]:
G_k = nx.read_weighted_edgelist('kamalahq_hashtag_network.edgelist')

In [5]:
G_t = nx.read_weighted_edgelist('teamtrump_hashtag_network.edgelist')

In [14]:
import json

In [69]:
nx.config.backend_priority = ["cugraph", "networkx"]

In [70]:
k_degree = nx.degree_centrality(G_k)

In [7]:
k_file_path = "k_degree.json"
with open(k_file_path, 'w') as json_file:
    json.dump(k_degree, json_file, indent=4)

In [8]:
t_degree = nx.degree_centrality(G_t)

In [9]:
t_file_path = "t_degree.json"
with open(t_file_path, 'w') as json_file:
    json.dump(t_degree, json_file, indent=4)

In [6]:
random.seed(42)
k_betweenness = nx.betweenness_centrality(G_k, k=50)

In [21]:
k_file_path = "k_betweenness.json"
with open(k_file_path, 'w') as json_file:
    json.dump(k_betweenness, json_file, indent=4)

In [9]:
random.seed(42)
t_betweenness = nx.betweenness_centrality(G_t, k=50)

In [22]:
t_file_path = "t_betweenness.json"
with open(t_file_path, 'w') as json_file:
    json.dump(t_betweenness, json_file, indent=4)

In [82]:
k_pagerank = nx.pagerank(G_k, weight='weight')

In [83]:
k_file_path = "k_pagerank.json"
with open(k_file_path, 'w') as json_file:
    json.dump(k_pagerank, json_file, indent=4)

In [84]:
t_pagerank = nx.pagerank(G_t, weight='weight')

In [85]:
t_file_path = "t_pagerank.json"
with open(t_file_path, 'w') as json_file:
    json.dump(t_pagerank, json_file, indent=4)

In [86]:
measures = ['betweenness', 'degree', 'pagerank']
k_measures_df = pd.DataFrame({})
for m in measures:
    with open(f'k_{m}.json') as f:
        data = json.load(f)
    df = pd.DataFrame(data.values(), index=data.keys(), columns = [m])
    k_measures_df = pd.concat([k_measures_df, df], axis=1)

In [87]:
measures = ['betweenness', 'degree', 'pagerank']
t_measures_df = pd.DataFrame({})
for m in measures:
    with open(f't_{m}.json') as f:
        data = json.load(f)
    df = pd.DataFrame(data.values(), index=data.keys(), columns = [m])
    t_measures_df = pd.concat([t_measures_df, df], axis=1)

In [88]:
k_measures_df.to_csv('kamala_nodes_measure.csv')
t_measures_df.to_csv('trump_nodes_measure.csv')

In [3]:
from networkx import approximation

In [92]:
t_avg_cluster = approximation.average_clustering(G_t, trials=100000, seed=42)
k_avg_cluster = approximation.average_clustering(G_k, trials=100000, seed=42)

In [93]:
t_avg_cluster, k_avg_cluster

(0.84527, 0.84437)

In [94]:
t_density = nx.density(G_t)
k_density = nx.density(G_k)

In [43]:
t_density, k_density

(0.00014269010099396336, 8.91057180921317e-05)

In [44]:
t_nnodes = nx.number_of_nodes(G_t)
k_nnodes = nx.number_of_nodes(G_k)

In [45]:
t_nnodes, k_nnodes

(246620, 487200)

In [46]:
t_nedges = nx.number_of_edges(G_t)
k_nedges = nx.number_of_edges(G_k)

In [64]:
t_nedges, k_nedges

(4339290, 10575216)